<a href="https://colab.research.google.com/github/QuyDuong106/INFOSYS722_BDAS/blob/main/PySpark/run_spark_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iteration 4 — saved native PySpark holdout run
This is the user's saved Colab Spark modelling notebook, moved into the canonical `PySpark` folder. Its model-result display is retained from the original execution. The setup cells now use the current repository structure; run all cells to produce a fresh result.

The workflow trains Spark logistic regression, random forest and GBT across five stratified 70/30 holdouts, with training-only three-fold Spark CV tuning. It uses all source predictors and standardizes continuous fields while preserving ordinal codes. It does not run the five-family Optuna/RFE20 experiments in `iteration4_BDAS/run_workflow.ipynb`.

Google Drive mounting is optional. Only missing Spark/display dependencies are installed; existing runtime packages are reused.


In [ ]:
USE_GOOGLE_DRIVE = False  # Optional; repository data is available without Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import sys, subprocess, importlib.util, shutil
from pathlib import Path
required = {'pyspark': 'pyspark>=3.5.3,<4.0',
            'sklearn': 'scikit-learn>=1.3', 'pandas': 'pandas>=2.0'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
java = shutil.which('java')
if not java or subprocess.run([java, '-version'], capture_output=True).returncode:
    raise RuntimeError('A working Java runtime is required for Spark.')


In [ ]:
from pathlib import Path
print('Notebook working directory:', Path.cwd())


In [ ]:
from pathlib import Path
import sys, subprocess
from importlib.metadata import version
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/INFOSYS722_BDAS')]
REPO_ROOT = next((p for p in candidates if
                 (p/'PySpark/spark_workflow.py').is_file() and
                 (p/'iteration4_BDAS/data/raw/dataset_gallstone').is_file()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path('/content/INFOSYS722_BDAS')
    if REPO_ROOT.exists():
        raise RuntimeError('The existing /content/INFOSYS722_BDAS folder lacks the current PySpark layout. Update its checkout or set REPO_ROOT to a current clone.')
    subprocess.check_call(['git', 'clone', 'https://github.com/QuyDuong106/INFOSYS722_BDAS.git', str(REPO_ROOT)])
SPARK_DIR = REPO_ROOT/'PySpark'
PROJECT_ROOT = REPO_ROOT/'iteration4_BDAS'  # Raw dataset remains with Iteration 4.
SPARK_RESULTS_DIR = SPARK_DIR/'results/spark_holdout'
sys.path.insert(0, str(SPARK_DIR))
for package in ['pyspark', 'scikit-learn', 'pandas']:
    print(package, version(package))
print('Spark code:', SPARK_DIR)
print('Results:', SPARK_RESULTS_DIR)


## Native Spark holdout execution
Run the model cell to train and save five-seed predictions, fitted Spark models and metrics under `PySpark/results/spark_holdout`. The saved result table below comes from the original Colab run; executing the model and display cells refreshes it. The module is now `PySpark/spark_workflow.py`.


In [7]:
from spark_workflow import create_spark, load_dataset, run_holdout
spark = create_spark()
frame, numeric, categorical = load_dataset(spark, PROJECT_ROOT/'data/raw/dataset_gallstone')
spark_metrics = run_holdout(spark, frame, numeric, categorical,
                            output_dir=SPARK_RESULTS_DIR, tune=True)

In [10]:
import json
import pandas as pd

# 1. Define path to metrics.json
json_file = SPARK_RESULTS_DIR / 'metrics.json'

# 2. Read and display the JSON content
if json_file.exists():
    with open(json_file, 'r') as f:
        metrics_data = json.load(f)

    # Display as a formatted pandas DataFrame for clear viewing
    if isinstance(metrics_data, list):
        metrics_df = pd.DataFrame(metrics_data)
    else:
        metrics_df = pd.DataFrame([metrics_data])

    display(metrics_df)
else:
    print(f"File not found at {json_file}")

,model,rows,TN,FP,FN,TP,accuracy,roc_auc,sensitivity,specificity,precision,f1,fnr,error_rate,seed
0,gbt,96,24,24,12,36,0.625000,0.729601,0.750000,0.500000,0.600000,0.666667,0.250000,0.375000,42
1,logistic,96,30,18,2,46,0.791667,0.858941,0.958333,0.625000,0.718750,0.821429,0.041667,0.208333,42
2,random_forest,96,31,17,18,30,0.635417,0.710937,0.625000,0.645833,0.638298,0.631579,0.375000,0.364583,42
3,gbt,96,33,15,8,40,0.760417,0.834635,0.833333,0.687500,0.727273,0.776699,0.166667,0.239583,7
4,logistic,96,34,14,5,43,0.802083,0.850260,0.895833,0.708333,0.754386,0.819048,0.104167,0.197917,7
5,random_forest,96,35,13,12,36,0.739583,0.835938,0.750000,0.729167,0.734694,0.742268,0.250000,0.260417,7
6,gbt,96,35,13,11,37,0.750000,0.814670,0.770833,0.729167,0.740000,0.755102,0.229167,0.250000,21
7,logistic,96,33,15,6,42,0.781250,0.855035,0.875000,0.687500,0.736842,0.800000,0.125000,0.218750,21
8,random_forest,96,35,13,10,38,0.760417,0.821181,0.791667,0.729167,0.745098,0.767677,0.208333,0.239583,21
9,gbt,96,34,14,13,35,0.718750,0.798611,0.729167,0.708333,0.714286,0.721649,0.270833,0.281250,84
